# Phase 1: Document Ingestion with Docling

This notebook demonstrates how to parse documents using Docling, chunk them semantically, generate embeddings, and store them in pgvector for retrieval.

## 1. Load Environment

In [ ]:
import os
import sys

import httpx
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

print(f"PGVECTOR_HOST: {os.getenv('PGVECTOR_HOST')}")
print(f"EMBEDDING_MODEL: {os.getenv('EMBEDDING_MODEL')}")
print("✅ Environment loaded")

## 2. Parse a Document with Docling

Docling's `DocumentConverter` automatically detects the file format and extracts structured content.

In [ ]:
from docling.document_converter import DocumentConverter

# Use a sample document — replace with your own PDF/DOCX path
sample_doc = os.path.join(os.path.dirname(os.getcwd()), "data", "sample_docs", "sample.pdf")

if not os.path.exists(sample_doc):
    # Download a sample PDF for testing
    import urllib.request
    os.makedirs(os.path.dirname(sample_doc), exist_ok=True)
    url = "https://arxiv.org/pdf/2408.09869"  # Docling technical report
    urllib.request.urlretrieve(url, sample_doc)
    print(f"Downloaded sample document to {sample_doc}")

converter = DocumentConverter()
result = converter.convert(sample_doc)
doc = result.document

print(f"Document: {os.path.basename(sample_doc)}")
print(f"Pages: {doc.num_pages() if hasattr(doc, 'num_pages') else 'N/A'}")
print(f"\nFirst 500 chars of markdown export:")
print(doc.export_to_markdown()[:500])
print("\n✅ Document parsed successfully")

## 3. Semantic Chunking

The `HybridChunker` splits documents while preserving structural boundaries (headings, tables, paragraphs) and respecting token limits.

In [ ]:
from docling.chunking import HybridChunker

chunker = HybridChunker(
    tokenizer="sentence-transformers/all-MiniLM-L6-v2",
    max_tokens=512,
)

chunks = list(chunker.chunk(doc))

print(f"Total chunks: {len(chunks)}")
print(f"\n--- Chunk 0 (first {min(200, len(chunks[0].text))} chars) ---")
print(chunks[0].text[:200])
if len(chunks) > 1:
    print(f"\n--- Chunk 1 (first {min(200, len(chunks[1].text))} chars) ---")
    print(chunks[1].text[:200])
print("\n✅ Document chunked successfully")

## 4. Generate Embeddings

Embed chunks using the Granite embedding model served via RHOAI.

In [ ]:
from openai import OpenAI

embedding_client = OpenAI(
    base_url=os.getenv("EMBEDDING_BASE_URL", "http://localhost:8000/v1"),
    api_key=os.getenv("EMBEDDING_API_KEY", "not-needed"),
    http_client=_http_client,
)

# Embed a small batch to verify
sample_texts = [c.text for c in chunks[:3]]
response = embedding_client.embeddings.create(
    input=sample_texts,
    model=os.getenv("EMBEDDING_MODEL", "granite-embedding-278m-multilingual"),
)

print(f"Embedding dimension: {len(response.data[0].embedding)}")
print(f"Embeddings generated for {len(response.data)} chunks")
print("✅ Embedding model working")

## 5. Store in pgvector

Insert all chunks with their embeddings into PostgreSQL+pgvector.

In [ ]:
import hashlib
import psycopg2
import psycopg2.extras

conn = psycopg2.connect(
    host=os.getenv("PGVECTOR_HOST", "localhost"),
    port=os.getenv("PGVECTOR_PORT", "5432"),
    dbname=os.getenv("PGVECTOR_DB", "doc_research"),
    user=os.getenv("PGVECTOR_USER", "postgres"),
    password=os.getenv("PGVECTOR_PASSWORD", "postgres"),
)
cur = conn.cursor()

document_id = hashlib.sha256(sample_doc.encode()).hexdigest()[:16]
document_name = os.path.basename(sample_doc)

# Register document
cur.execute(
    """INSERT INTO documents (id, name, file_type, chunk_count, status)
       VALUES (%s, %s, %s, %s, 'processing')
       ON CONFLICT (id) DO UPDATE SET status = 'processing', updated_at = NOW()""",
    (document_id, document_name, os.path.splitext(sample_doc)[1], len(chunks)),
)

# Clear existing chunks
cur.execute("DELETE FROM document_chunks WHERE document_id = %s", (document_id,))

# Embed and insert all chunks in batches
batch_size = 32
total_inserted = 0

for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    texts = [c.text for c in batch]
    
    resp = embedding_client.embeddings.create(
        input=texts,
        model=os.getenv("EMBEDDING_MODEL", "granite-embedding-278m-multilingual"),
    )
    
    for idx, (chunk, emb_data) in enumerate(zip(batch, resp.data)):
        cur.execute(
            """INSERT INTO document_chunks (document_id, document_name, chunk_index, content, embedding)
               VALUES (%s, %s, %s, %s, %s::vector)""",
            (document_id, document_name, i + idx, chunk.text, str(emb_data.embedding)),
        )
        total_inserted += 1
    
    print(f"  Inserted batch {i//batch_size + 1} ({total_inserted}/{len(chunks)} chunks)")

# Mark document as completed
cur.execute(
    "UPDATE documents SET status = 'completed', chunk_count = %s, updated_at = NOW() WHERE id = %s",
    (total_inserted, document_id),
)

conn.commit()
cur.close()
conn.close()

print(f"\nDocument '{document_name}' indexed: {total_inserted} chunks")
print("✅ Chunks stored in pgvector")

## 6. Verify Storage

Confirm the document and chunks are properly stored.

In [ ]:
conn = psycopg2.connect(
    host=os.getenv("PGVECTOR_HOST", "localhost"),
    port=os.getenv("PGVECTOR_PORT", "5432"),
    dbname=os.getenv("PGVECTOR_DB", "doc_research"),
    user=os.getenv("PGVECTOR_USER", "postgres"),
    password=os.getenv("PGVECTOR_PASSWORD", "postgres"),
)
cur = conn.cursor()

cur.execute("SELECT id, name, status, chunk_count FROM documents")
docs = cur.fetchall()
print("Indexed documents:")
for d in docs:
    print(f"  {d[0]} | {d[1]} | {d[2]} | {d[3]} chunks")

cur.execute("SELECT COUNT(*) FROM document_chunks")
count = cur.fetchone()[0]
print(f"\nTotal chunks in database: {count}")

cur.close()
conn.close()
print("✅ Verification complete")